# 유방암 데이터셋

---

## 데이터셋의 목적

이 데이터셋의 주 목적은 **유방 종양의 미세침 흡인 세포(FNA) 이미지에서 추출한 특징들을 바탕으로, 해당 종양이 악성(Malignant)인지 양성(Benign)인지 예측**하는 것입니다.


* **`malignant` (악성, 암이 맞음):** 보통 머신러닝에서 레이블 `0`으로 표현됩니다.
* **`benign` (양성, 단순 종양):** 보통 머신러닝에서 레이블 `1`으로 표현됩니다.

> ⚠️ **주의:** 사이킷런 데이터셋에서는 특이하게 **양성(Benign)이 1, 악성(Malignant)이 0**으로 매핑되어 있습니다. 보통 질병 유무를 따질 때 질병이 있는 상태를 `1`로 두는 경우가 많아 헷갈리기 쉬우니 데이터 분석 시 꼭 확인해야 합니다.

---

## 데이터 구조 및 스펙

전체적인 데이터의 크기와 구조는 다음과 같습니다.

* **전체 샘플 수 (행, Rows):** 569개
* 악성(Malignant): 212개
* 양성(Benign): 357개


* **특징 수 (열, Columns/Features):** 30개

---

## 30개의 특징(Features)은 어떻게 만들어졌을까?

종양 세포 핵의 디지털 이미지에서 **10가지 핵심 측정값**을 컴퓨터로 계산해 낸 것입니다.

| 핵심 측정값 (10개) | 설명 |
| --- | --- |
| **`radius`** | 중심에서 외곽선까지의 거리 평균 |
| **`texture`** | 회색조(Gray-scale) 명암 값의 표준편차 |
| **`perimeter`** | 종양의 둘레 길이 |
| **`area`** | 종양의 면적 |
| **`smoothness`** | 반경 길이의 국소적 변화 (부드러운 정도) |
| **`compactness`** | $\frac{\text{perimeter}^2}{\text{area}} - 1.0$ (조밀한 정도) |
| **`concavity`** | 윤곽선에서 오목한 부분의 심한 정도 |
| **`concave points`** | 윤곽선 중 오목한 부분의 개수 |
| **`symmetry`** | 대칭성 |
| **`fractal dimension`** | 프랙탈 차원 (외곽선의 복잡도) |

이 **10가지 측정값** 각각에 대해 통계적인 수치인 **평균(mean), 표준오차(error), 가장 큰 값의 평균(worst)** 3가지를 곱하여 총 30개($10 \times 3$)의 피처가 구성됩니다.

* *예시: `mean radius`, `radius error`, `worst radius` ... 이런 식으로 30개가 존재합니다.*


In [6]:
from sklearn.datasets import load_breast_cancer

# ── 데이터 로드 ──────────────────────────────────────────────────
data = load_breast_cancer()
print("target_names:", data.target_names)
# ['malignant' 'benign']  ← index 0=악성, index 1=정상

# ⚠ 주의: target=0이 malignant(악성), target=1이 benign(정상)!
# 직관과 반대 → y = 1 - data.target 으로 뒤집어서 사용
# y=1: 악성(malignant), y=0: 정상(benign)
y_cancer = 1 - data.target
X_cancer = data.data

target_names: ['malignant' 'benign']


유방암 실습 계획

데이터 특징 먼저 확인
전체: 569개
악성(Malignant): 212개 → y=1 (뒤집은 후)
양성(Benign):    357개 → y=0 (뒤집은 후)
특성: 30개

핵심 포인트
원본: 악성=0, 양성=1  ← 직관과 반대
뒤집기: y = 1 - data.target
뒤집은 후: 악성=1, 양성=0  ← 질병 있음=1로 통일

할 것
1. 데이터 불러오기 + label 방향 확인
2. y = 1 - data.target 적용
3. stratify=y 분리
4. 로지스틱 회귀 학습
5. confusion_matrix → FN 확인
6. classification_report
7. AUC
8. 임계값 0.3 적용 → Recall 변화 확인
9. "왜 Recall이 올라가는지" 1~2문장 설명

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

In [7]:
# 1. 데이터 불러오기
data = load_breast_cancer()
print("클래스 이름:", data.target_names)  # 확인

# 2. label 뒤집기
X = data.data
y = 1 - data.target   # 악성=1, 양성=0

print("악성(1):", y.sum(), "명")
print("양성(0):", len(y) - y.sum(), "명")

클래스 이름: ['malignant' 'benign']
악성(1): 212 명
양성(0): 357 명


라벨을 뒤집는 이유:
악성을 1로 만들어야 FN = 악성인데 정상으로 예측한 것 으로 올바르게 계산되기 때문입니다.
악성이 0이면 Recall이 양성 기준으로 계산되어서 우리가 줄이고 싶은 FN이 제대로 반영이 안 됩니다. 악성을 1로 뒤집어야 Recall이 악성 기준으로 계산됩니다.

In [8]:
# train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

print("훈련:", X_train.shape)
print("테스트:", X_test.shape)

훈련: (426, 30)
테스트: (143, 30)
